# Missing Data Matrix — 30 Model Variables

`missingno.matrix()` restricted to the 33 independent variables used in modelling.  
Dark = present, white = missing. Right sparkline = row-level completion rate.

In [1]:
import os
import pandas as pd
import missingno as msno
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

BASE_DIR    = os.path.abspath(os.path.join('..', '..'))   # notebooks/eda/ -> repo root
DATA_PATH   = os.path.join(BASE_DIR, 'data', 'final', 'model_dataset.csv')
OUTPUT_DIR  = os.path.join(BASE_DIR, 'figures')
MATRIX_PNG  = os.path.join(OUTPUT_DIR, 'missing_matrix.png')
os.makedirs(OUTPUT_DIR, exist_ok=True)

variables = [
    # Episode rhythm features
    'episode_duration_sec', 'episode_beat_count', 'episode_rr_cv',
    # HR during episode duration
    'hr_dur_mean', 'hr_dur_std', 'hr_dur_min', 'hr_dur_slope',
    # SpO2 during episode duration
    'spo2_dur_mean', 'spo2_dur_std', 'spo2_dur_min', 'spo2_dur_slope',
    # MAP during episode duration
    'map_dur_mean', 'map_dur_std', 'map_dur_min', 'map_dur_slope',
    # Patient-level arrhythmia burden
    'total_episode_count', 'total_arrhythmia_burden_sec',
    'longest_episode_duration_sec', 'first_episode_start_sec',
    # Clinical
    'age', 'bmi', 'asa', 'preop_htn', 'preop_dm',
    'preop_hb', 'preop_plt', 'preop_na', 'preop_k',
    'preop_cr',
    'preop_alb'
]

In [2]:
df = pd.read_csv(DATA_PATH)
missing_vars = [v for v in variables if v not in df.columns]
if missing_vars:
    print(f'WARNING — not found, skipping: {missing_vars}')
use_cols = [v for v in variables if v in df.columns]
data = df[use_cols].copy()
print(f'Using {len(use_cols)} variables, {df.shape[0]} episodes')
miss_pct = data.isna().mean().mul(100)
print(f'Columns with any missing: {(miss_pct > 0).sum()}')
print(miss_pct[miss_pct > 0].sort_values(ascending=False).to_string())

Using 30 variables, 1284 episodes
Columns with any missing: 24
episode_rr_cv     36.838006
preop_na          17.056075
preop_k           17.056075
map_dur_mean      15.420561
map_dur_std       15.420561
map_dur_min       15.420561
map_dur_slope     15.420561
preop_cr          13.785047
preop_alb         13.395639
preop_hb          13.006231
preop_plt         12.616822
asa                9.423676
spo2_dur_mean      7.554517
spo2_dur_std       7.554517
spo2_dur_min       7.554517
spo2_dur_slope     7.554517
hr_dur_slope       6.931464
hr_dur_mean        6.931464
hr_dur_std         6.931464
hr_dur_min         6.931464
bmi                6.853583
age                6.853583
preop_htn          6.853583
preop_dm           6.853583


In [3]:
# Missingness summary table figure
TABLE_PNG = os.path.join(OUTPUT_DIR, 'missingness_table.png')
TABLE_CSV = os.path.join(OUTPUT_DIR, 'missingness_table.csv')

miss_n    = data.isna().sum()
miss_pct  = data.isna().mean() * 100
complete_n = data.notna().sum()

table_df = pd.DataFrame({
    'Variable':     list(data.columns),
    'Missing (n)':  miss_n.values,
    'Missing (%)':  miss_pct.round(1).values,
    'Complete (n)': complete_n.values,
}).sort_values('Missing (%)', ascending=False).reset_index(drop=True)

table_df.to_csv(TABLE_CSV, index=False)
print(f'Saved CSV: {TABLE_CSV}')

n_rows = len(table_df)
fig, ax = plt.subplots(figsize=(11, n_rows * 0.38 + 1.6))
ax.axis('off')

col_labels = ['Variable', 'Missing (n)', 'Missing (%)', 'Complete (n)']
cell_text  = []
for _, row in table_df.iterrows():
    cell_text.append([
        row['Variable'],
        str(int(row['Missing (n)'])),
        f"{row['Missing (%)']:.1f}%",
        str(int(row['Complete (n)'])),
    ])

tbl = ax.table(
    cellText=cell_text,
    colLabels=col_labels,
    loc='center',
    cellLoc='center',
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1, 1.5)

# Header row styling
for j in range(len(col_labels)):
    tbl[0, j].set_facecolor('#2c5f8a')
    tbl[0, j].set_text_props(color='white', fontweight='bold')

# Row colour by missingness level
for i, row in table_df.iterrows():
    pct = row['Missing (%)']
    if pct == 0:
        colour = '#eaf4ea'   # green  — complete
    elif pct < 10:
        colour = '#fffde7'   # yellow — low
    elif pct < 20:
        colour = '#fff3e0'   # orange — moderate
    else:
        colour = '#fce4ec'   # red    — high
    for j in range(len(col_labels)):
        tbl[i + 1, j].set_facecolor(colour)

# Column widths
tbl.auto_set_column_width([0, 1, 2, 3])

ax.set_title(
    f'Missing Data Summary — {n_rows} Model Variables  ({len(data)} episodes)',
    fontsize=12, pad=16, fontweight='bold'
)
plt.tight_layout()
fig.savefig(TABLE_PNG, dpi=200, bbox_inches='tight')
plt.close(fig)
print(f'Saved PNG: {TABLE_PNG}')


Saved CSV: c:\Users\sukka\Downloads\ioh-prediction\figures\missingness_table.csv


Saved PNG: c:\Users\sukka\Downloads\ioh-prediction\figures\missingness_table.png


In [4]:
ax = msno.matrix(
    data,
    figsize=(16, 8),
    sparkline=True,
    fontsize=9,
    color=(0.25, 0.45, 0.65),
    labels=True
)
fig = ax.get_figure()
fig.suptitle(
    f'Missing Data Matrix — {len(use_cols)} Model Variables  ({df.shape[0]} episodes)',
    fontsize=13, y=1.02
)
for tick in ax.get_xticklabels():
    tick.set_rotation(45)
    tick.set_ha('right')

fig.savefig(MATRIX_PNG, dpi=300, bbox_inches='tight')
plt.close(fig)
print(f'Saved: {MATRIX_PNG}')

Saved: c:\Users\sukka\Downloads\ioh-prediction\figures\missing_matrix.png
